In [1]:
%load_ext autoreload
%autoreload 2

import edgar

from equibot.config import settings
from equibot.parsers import transcript
from etl.db import connect


edgar.set_identity(settings.sec_api_username)

with connect() as conn:                      # closes the connection on exit
    print(conn.execute("SELECT version()").fetchone()["version"])

c:\Users\benak\repos\EquiBot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PostgreSQL 18.6 (Debian 18.6-1.pgdg13+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


In [2]:
from etl.upserts import *

snow = edgar.Company("SNOW")
filings = snow.get_filings(form="10-K")

In [3]:
for filing in filings:
    upsert_filing(company=snow, filing=filing)

RecursionError: maximum recursion depth exceeded

In [6]:
upsert_filing(snow, filings[0])

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 983.33it/s]


1

In [11]:
goog = edgar.Company("GOOG")
filings = goog.get_filings(form="10-Q")

In [14]:
# add company
upsert_company(goog)

8

In [15]:
upsert_filing(goog, filings[0])

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 1364.62it/s]
C:\Users\benak\repos\laya\laya\router.py:999: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(repo, **kwargs)


6

In [35]:
# ingest company
from edgar.entity.core import Company

def upsert_company(company: Company, replace_entry: bool = False) -> int:
    """Update ref.company table with the given company information.
    
    Idempotent operation unless `replace_entry` is set to True.
    Returns:
        int: The `company_id` of the inserted or updated company.
    """
    ticker = company.tickers[0]
    cik = company.cik
    name = company.name
    industry = company.industry
    fiscal_year_end = company.fiscal_year_end

    with connect() as conn:
        cur = conn.cursor()
        if replace_entry:
            cur.execute(
                """
                REPLACE INTO ref.company (ticker, cik, name, industry, fiscal_year_end)
                VALUES (%s, %s, %s, %s, %s)
                RETURNING company_id
                """,
                (ticker, cik, name, industry, fiscal_year_end),
            )
        else:
            cur.execute(
                # uses `DO UPDATE` on conflict so that we can return `company_id`
                # `SET` variables are mostly arbitrary, but all can change so can be worth updating
                """
                INSERT INTO ref.company (ticker, cik, name, industry, fiscal_year_end)
                VALUES (%s, %s, %s, %s, %s)
                ON CONFLICT (ticker) DO UPDATE SET 
                    name = EXCLUDED.name
                    , industry = EXCLUDED.industry
                    , fiscal_year_end = EXCLUDED.fiscal_year_end
                RETURNING company_id
                """,
                (ticker, cik, name, industry, fiscal_year_end),
            )
        
        company_id = cur.fetchone()["company_id"]
    
    return company_id

In [52]:
company_id = upsert_company(snow, replace_entry=False)

In [48]:
from datetime import date

def _resolve_fiscal_period(company: Company,filing: EntityFiling) -> str:
    """Resolve the fiscal period from form type, period end, and fiscal year end"""
    period_end = filing.period_of_report
    form_type = filing.form
    fiscal_year_end = company.fiscal_year_end

    if isinstance(period_end, str):
        period_end = date.fromisoformat(period_end)

    fye_month = int(fiscal_year_end[:2])
    fye_day = int(fiscal_year_end[2:])

    # Infer the fiscal year end date that follows (or equals) period_end
    fye_year = period_end.year
    if (period_end.month, period_end.day) > (fye_month, fye_day):
        fye_year += 1

    # Months elapsed since the fiscal year start, based on the months between
    # period_end and fiscal year end (0 months remaining -> last quarter)
    months_to_fye = (fye_year - period_end.year) * 12 + (fye_month - period_end.month)
    months_since_start = 12 - months_to_fye  # 1..12

    quarter = (months_since_start - 1) // 3 + 1
    quarter_label = f"Q{quarter}"

    if form_type == "10-K":
        return "FY"
    elif form_type == "10-Q":
        return quarter_label


In [50]:
company = snow

ticker = company.tickers[0]
cik = company.cik
name = company.name
industry = company.industry
fiscal_year_end = company.fiscal_year_end

accession_number = filing.accession_number
form_type = filing.form
period_end = filing.period_of_report
fiscal_period = _resolve_fiscal_period(company, filing)
published_at = filing.filing_date
source_url = filing.filing_url
parser_version = "1.0"


print(accession_number, form_type, period_end, fiscal_period, published_at, source_url, parser_version)

0001640147-26-000008 10-K 2026-01-31 FY 2026-03-20 https://www.sec.gov/Archives/edgar/data/1640147/000164014726000008/snow-20260131.htm 1.0


In [ ]:
from edgar.entity import EntityFiling

PARSER_VERSION = "1.0"

def insert_filing(company: Company, filing: EntityFiling, company_id: int | str) -> int:
    """Insert filing into `raw.filing` table

    Returns:
        int: The `filing_id` of the inserted filing.
    """
    fiscal_year = company.fiscal_year_end

    accession_no = filing.accession_number
    form_type = filing.form
    period_end = filing.period_of_report
    fiscal_period = _resolve_fiscal_period(company, filing)
    published_at = filing.filing_date
    source_url = filing.filing_url
    parser_version = PARSER_VERSION

    # insert the filing into `raw.filing`
    with connect() as conn:
        cur = conn.cursor()
        cur.execute(
            """
            INSERT INTO raw.filing (
                company_id , accession_no , form_type , period_end
                , fiscal_year , fiscal_period , published_at , source_url
                , parser_version
            )
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (accession_no) DO UPDATE SET period_end = EXCLUDED.period_end
            RETURNING filing_id
            """,
            (
                company_id, accession_no, form_type, period_end,
                fiscal_year, fiscal_period, published_at, source_url,
                parser_version,
            ),
        )
        return cur.fetchone()['filing_id']

In [58]:
filing_id = insert_filing(company, filing, company_id)

In [60]:
SECTION_KEYS = {
    "business": "business",
    "risk_factors": "risk_factors",
    "mdna": "management_discussion"
}

def insert_section(filing, filing_id: int | str, section: str) -> int:
    """Insert section into raw.filing_section"""
    
    pass

In [109]:
from pydantic_ai.models.openrouter import OpenRouterModel
from pydantic_ai.providers.openrouter import OpenRouterProvider

model = OpenRouterModel(
    "z-ai/glm-5.3-flash",
    provider=OpenRouterProvider(api_key=settings.openrouter_api_key),
)

In [116]:
from pydantic_ai import Agent

agent = Agent(model=model)
result = await agent.run("How much wood can a woodchuck chuck?")
print(result.output)

Ah, the classic tongue twister! The traditional answer is: "A woodchuck would chuck as much wood as a woodchuck could chuck if a woodchuck could chuck wood."

Of course, in reality, woodchucks (also known as groundhogs) don't actually chuck wood at all—they're burrowing rodents that dig tunnels and eat vegetation. But if you want a semi-scientific answer: wildlife expert Richard Thomas famously calculated in 1988 that, based on the volume of dirt a woodchuck moves when digging its burrow, a woodchuck could hypothetically chuck about 700 pounds of wood.

So there you have it—somewhere between "none" and "roughly 700 pounds," depending on how seriously you take the question!


In [119]:
tenq.form

'10-Q'

In [120]:
res = {}
for s in tenq.sections:
    res[s] = await agent.run(
        f"The following is a section of a {tenq.form} financial report with item number {s}:\n"
        f"Please read the section and provide a title for it along with a brief summary.\n"
        f"Ex. 'Title: Management's Discussion and Analysis, Summary: This section provides an overview of the company's financial performance and key metrics.'\n"
        f"Section Text:\n{tenq.sections.get(s).text()}"
    )

In [121]:
for r in res:
    print(f"Section: {r}")
    print(f"Title and Summary: {res[r].output}\n")
    print(f"Usage: {res[r].usage}\n\n")


Section: part_i_item_1
Title and Summary: Title: Financial Statements (Unaudited)

Summary: This section presents Snowflake Inc.'s unaudited condensed consolidated financial statements for the quarterly period ended July 31, 2026, comprising the balance sheets, statements of operations, comprehensive loss, stockholders' equity, and cash flows, together with accompanying notes. Key figures include Q2 revenue of $1.55 billion (up from $1.14 billion in the prior-year quarter) and a net loss of $191.7 million (improved from $297.9 million), with six-month revenue of $2.94 billion and total assets of $8.7 billion. The notes disclose significant accounting policies, revenue and RPO details (approximately $9.0 billion in remaining performance obligations), recent business acquisitions (Natoma Labs and Observe, Inc.), $2.3 billion of 0% convertible senior notes, stock-based compensation plans, income taxes, and ongoing legal proceedings including data breach-related litigation.

Usage: RunUsag